# Working with Futures

We can look into threads first, wich behave as in Java.

## Threads first

A `Thread` a lightweight sub-process executing a part of a program.

A `Thread` in scala is really just a Java Thread. The Java Virtual Machine allows an application to have multiple threads of execution running concurrently.

Threads with higher priority are executed in preference to threads with lower priority. When a JVM starts up, there is usually a single non-daemon thread. The JVM continues to execute threads until either of the following occurs:

 - The exit method of class Runtime has been called.
 - All threads that are not daemon threads have died.

There are two ways to create a new thread of execution. One is to declare a class to be a subclass of `Thread`. This subclass should override the `run` method of class `Thread`. An instance of the subclass can then be allocated and started. 

In [1]:
class SomeThread extends Thread :
        override def run = println("hi there")

defined class SomeThread

In the same way we can create ananonymous instatiation of the `Thread` class:

In [3]:
val t = new Thread {
    override def run = println("hi there")
}

t: Thread = Thread[Thread-1,5,main]

Once the thread is instatiated it can be started.

In [4]:
t.start

hi there

And we can check if it is still alive.

In [5]:
t.isAlive

res4: Boolean = false

The other way to create a thread is to declare a class that implements the `Runnable` interface. That class then implements the `run` method. An instance of the class can then be allocated, passed as an argument when creating `Thread`, and started. The same example in this other style looks like the following:

In [6]:
val r = new Runnable {
    def run = println("now runnable")
}

val t1 = new Thread(r).start

now runnable

r: Runnable = ammonite.$sess.cmd5$$anon$1@42e15046

## Thread lifecyle

![lala](img/lifecycle.png)

## Coding with threads

Code inside the run method can take a longer time.

In [7]:
val t = new Thread {
    override def run = 
        println("hi there")
        Thread.sleep(10000)
}

t: Thread = Thread[Thread-3,5,main]

Before starting the thread we can check if it is alive or not.

In [8]:
t.isAlive

res7: Boolean = false

Nothing will happen if we don't start the thread first.

In [9]:
t.start

hi there

In [10]:
t.isAlive

res9: Boolean = true

With the `join` method we can wait for the thread to terminate and join the main one.

In [11]:
t.join
t.isAlive

res10_1: Boolean = false

The thread can run for an indefinite lapse of time.

In [12]:
val t = new Thread {
    override def run=
        while true do
            println("working")
            Thread.sleep(5000)
}

t: Thread = Thread[Thread-4,5,main]

In [13]:
t.start

working

While the thread is alive we can still check at its properties.

In [15]:
t.isAlive
Thread.sleep(10000)

working
working


res14_0: Boolean = true

Or at some pointw we can interrupt it or stop it. Once stopped, the thread cannot be restarted. We would need to instantiate a new one.

In [16]:
t.interrupt

Exception in thread "Thread-4" 

In [17]:
t.isAlive


res16: Boolean = false

In [18]:
t.stop

In [19]:
t.isAlive

res18: Boolean = false

## Synchronous and Asynchronous execution

Oftentimes we need to implement and execute code that can take variable amounts of time. Idle execution times can render our code slower and more ineffective. 

In [20]:
import scala.util.Random

import scala.util.Random


Let's declare a method that takes a ahort time to throw a result (e.g., a random number).

In [21]:
def shortOperation =
    Thread.sleep(10)
    Random.nextInt(5)

defined function shortOperation

When we call the short operation once.

In [22]:
shortOperation

res21: Int = 3

Or we can call it multiple times. As they are short operations all operations seems to be executed instantaneously. 

In [23]:
shortOperation
shortOperation
shortOperation

res22_0: Int = 3
res22_1: Int = 1
res22_2: Int = 0

All of these operations are executed synchronously, i.e., every short operation is only invoked after the previous one has ended. This synchrony is more visible for a longer operation like the following:

In [24]:
def longOperation =
    println("starting long operation")
    Thread.sleep(5000)
    val result = Random.nextInt(5)
    println(s"completed: $result")
    result

defined function longOperation

Now if we call the short operation after the long one, we will first have to wait a number of seconds, until the `longOperation` is over.

In [26]:
longOperation
shortOperation

starting long operation
completed: 0


res25_0: Int = 0
res25_1: Int = 3

Using threads we can envelop the long operations so that each of them can be inside the `run` method. 

In [30]:
val t1 = new Thread {override def run:Unit = longOperation}
val t2 = new Thread {override def run:Unit = longOperation}

t1: Thread = Thread[Thread-7,5,main]
t2: Thread = Thread[Thread-8,5,main]

After starting the two threads, we can see that both long operations are run in parallel, eventually finishing their computation. 

In [29]:
t1.start
t2.start
Thread.sleep(6000)

: 